# Esplorazione del Database Simbad

## Introduzione

**Simbad** (Set of Identifications, Measurements, and Bibliography for Astronomical Data) è un database astronomico gestito dal Centre de Données astronomiques de Strasbourg (CDS). Contiene informazioni su oltre 10 milioni di oggetti celesti: stelle, galassie, ammassi, nebulose, esopianeti, ecc.

Caratteristiche principali:
- **Identificatori multipli**: uno stesso oggetto può essere referenziato con nomi diversi (HD, HIP, BD, SAO, ...)
- **Coordinate equatoriali**: RA (Ascensione Retta) e Dec (Declinazione)
- **Classificazione**: tipo spettrale, morfologia, classe di oggetto
- **Misure fotometriche**: magnitudini in varie bande
- **Bibliografia**: riferimenti alle pubblicazioni che menzionano l'oggetto

In questo notebook esploreremo Simbad tramite la libreria Python `astroquery.simbad`.

In [ ]:
from astroquery.simbad import Simbad
import pandas as pd

## 1. Query su un oggetto specifico: HD 209458

HD 209458 è una stella di tipo G0V nella costellazione di Pegaso, nota per ospitare l'esopianeta **HD 209458 b** (Osiride), scoperto nel 1999. È stato il primo pianeta extrasolare osservato con il metodo del transito e il primo di cui è stata rilevata l'atmosfera.

In [ ]:
simbad = Simbad()
result = simbad.query_object("HD 209458")
print(result)

## 2. Informazioni dettagliate con campi aggiuntivi

Simbad permette di aggiungere campi informativi oltre a quelli base. Aggiungiamo tipo spettrale, parallasse, velocità radiale e tipo di oggetto.

In [ ]:
Simbad.reset_votable_fields()
Simbad.add_votable_fields('sptype', 'plx', 'rv_value', 'otype', 'flux(V)')

result_detailed = simbad.query_object("HD 209458")
if result_detailed is not None:
    df_hd = result_detailed.to_pandas()
    print("=== Informazioni dettagliate per HD 209458 ===\n")
    for col in df_hd.columns:
        val = df_hd[col].values[0]
        print(f"{col}: {val}")

## 3. Ricerca per coordinate (query regionale)

Simbad consente di cercare oggetti in una regione di cielo specificata da coordinate e raggio. Cerchiamo oggetti vicini a HD 209458.

In [ ]:
result_region = simbad.query_region(
    "22 03 10.7727 +18 53 03.549", radius="0.1d"
)
if result_region is not None:
    df_region = result_region.to_pandas()
    print(f"Oggetti trovati nel raggio di 0.1°: {len(df_region)}\n")
    display(df_region[['MAIN_ID', 'RA', 'DEC', 'OTYPE']].head(10))

## 4. Ricerca per criteri: stelle di tipo solare

Cerchiamo stelle con tipo spettrale G2V (simili al Sole) con velocità radiale nota.

In [ ]:
Simbad.reset_votable_fields()
Simbad.add_votable_fields('sptype', 'rv_value')

result_g2v = simbad.query_criteria(
    "sptype='G2V' & rv_value>0", limit=10
)
if result_g2v is not None:
    df_g2v = result_g2v.to_pandas()
    print("Stelle di tipo G2V con velocità radiale positiva:")
    display(df_g2v[['MAIN_ID', 'SP_TYPE', 'RV_VALUE']])

## 5. Query su un oggetto con più identificatori: stella multipla

Simbad gestisce identificatori multipli. Proviamo con un oggetto ben noto.

In [ ]:
identifiers = ["Sirius", "HD 48915", "HIP 32349", "Alpha CMa"]

for ident in identifiers:
    res = simbad.query_object(ident)
    if res is not None:
        print(f"{ident} trovato: {res['MAIN_ID'][0]}  RA={res['RA'][0]}  DEC={res['DEC'][0]}")

## Conclusioni

Simbad è uno strumento versatile per la ricerca di informazioni astronomiche:

- **query_object()**: ricerca per nome/identificatore
- **query_region()**: ricerca per coordinate e raggio
- **query_criteria()**: ricerca per criteri specifici (tipo spettrale, magnitudine, ecc.)
- **add_votable_fields()**: campi aggiuntivi personalizzabili

Per maggiori informazioni: https://simbad.cds.unistra.fr/